In [9]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from typing import List, Tuple
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


device = torch.device("cuda")


class MetalDataset(Dataset):
    def __init__(self, texts: List[str], numericals: List[float], labels: List[int]):
        self.tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
        self.texts = texts
        self.numericals = numericals
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Токенизация текста (возвращаем тензоры с batch dim=1)
        text_inputs = self.tokenizer(self.texts[idx], padding="max_length", truncation=True, max_length=512, return_tensors='pt')
        input_ids = text_inputs["input_ids"].squeeze(0)         # размер [seq_len]
        attention_mask = text_inputs["attention_mask"].squeeze(0) # размер [seq_len]

        # Числовой признак - скалярный тензор
        num_data = torch.tensor(self.numericals[idx], dtype=torch.float32)

        # Метка класса
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return {"input_ids": input_ids, "attention_mask": attention_mask, "numerical_input": num_data}, label

def collate_fn(batch):
    # batch — список элементов вида (inputs_dict, label)
    inputs = {
        "input_ids": torch.stack([item[0]['input_ids'] for item in batch]),          # [batch_size, seq_len]
        "attention_mask": torch.stack([item[0]['attention_mask'] for item in batch]),# [batch_size, seq_len]
        "numerical_input": torch.stack([item[0]['numerical_input'] for item in batch]) # [batch_size]
    }
    labels = torch.stack([item[1] for item in batch])  # [batch_size]
    return inputs, labels

Задача - по спаршенным данным с Яндекс карт, а именно - тексту объявлений в карточке субъекта и цене определить картегорию товара. Проводилась полуавтоматическая разметках данных, рассматривались только уникальные сочетания текста и стоимости

In [10]:
df = pd.read_excel('categs.xlsx')
df = df.dropna()

In [11]:
labelencoder = LabelEncoder()

df.loc[:, 'category'] = labelencoder.fit_transform(df.loc[:, 'category'])
df

,name,price,category
0,"""оборонка"" советская от /кг",100.0,18
1,"(10%ni), не габарит нержавеющая сталь ni: 9,8%...",65.0,20
2,(10pee) 100% новый hvled815pftr hvled815pf hvl...,943.0,18
3,(12а) жесть (сталь толщиной менее 4 мм),11.0,0
4,(12а2) жесть оцинкованная,11.0,0
...,...,...,...
15569,электротехнический за кг,110.0,8
15570,электротехнический кабель,140.0,8
15572,"электротехнический лом алюминия, проводники то...",120.0,8
15573,"электротехнический, проводники",174.0,8


In [12]:
unique_cats = len(df['category'].unique())

In [22]:
df['category_counts'] = df.groupby(['category'])['name'].transform('count')

In [29]:
df = df[df['category_counts'] > 3] # Малые классы убираем

In [32]:
df['price'] = df['price'].astype(float)

X = df.drop(['category'], axis=1)
y = df[['category']]

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

name = list(Xtrain['name'])
price = list(Xtrain['price'])
categories = list(ytrain['category'])

name_val = list(Xtest['name'])
price_val = list(Xtest['price'])
categories_cal = list(ytest['category'])

dataset = MetalDataset(name, price, categories)
val_dataset = MetalDataset(name_val, price_val, categories_cal)

dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

C:\Users\Илья\AppData\Local\Temp\ipykernel_6496\338119925.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['price'] = df['price'].astype(float)


In [33]:
# Определение моделей
class TextBranch(nn.Module):
    def __init__(self):
        super(TextBranch, self).__init__()
        self.model = AutoModel.from_pretrained("bert-base-cased")

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state_cls = outputs.last_hidden_state[:, 0, :]  # CLS токен
        return last_hidden_state_cls

class NumericalBranch(nn.Module):
    def __init__(self, input_dim):
        super(NumericalBranch, self).__init__()
        self.fc = nn.Linear(input_dim, 64)

    def forward(self, x):
        return self.fc(x.unsqueeze(1))  # Добавляем размерность для Linear

class CombinedModel(nn.Module):
    def __init__(self):
        super(CombinedModel, self).__init__()
        self.text_branch = TextBranch()
        self.numerical_branch = NumericalBranch(input_dim=1)
        self.classifier = nn.Sequential(
            nn.Linear(768 + 64, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, unique_cats)  # количество классов
        )

    def forward(self, inputs):
        text_output = self.text_branch(inputs["input_ids"], inputs["attention_mask"])
        numerical_output = self.numerical_branch(inputs["numerical_input"])
        combined_output = torch.cat((text_output, numerical_output), dim=-1)
        output = self.classifier(combined_output)
        return output

In [34]:
# Обучение
model = CombinedModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

for epoch in range(10):
    model.train()
    for batch_idx, (data, target) in enumerate(dataloader):
        data = {k: v.to(device) for k, v in data.items()}
        target = target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        print(f"Epoch [{epoch+1}/10], Batch [{batch_idx+1}/{len(dataloader)}], Loss: {loss.item():.4f}")

Epoch [1/10], Batch [1/603], Loss: 39.0159
Epoch [1/10], Batch [2/603], Loss: 33.6203
Epoch [1/10], Batch [3/603], Loss: 16.3805
Epoch [1/10], Batch [4/603], Loss: 13.5848
Epoch [1/10], Batch [5/603], Loss: 14.9447
Epoch [1/10], Batch [6/603], Loss: 16.2703
Epoch [1/10], Batch [7/603], Loss: 10.9374
Epoch [1/10], Batch [8/603], Loss: 25.3358
Epoch [1/10], Batch [9/603], Loss: 16.6144
Epoch [1/10], Batch [10/603], Loss: 15.4839
Epoch [1/10], Batch [11/603], Loss: 13.3159
Epoch [1/10], Batch [12/603], Loss: 9.7063
Epoch [1/10], Batch [13/603], Loss: 13.5611
Epoch [1/10], Batch [14/603], Loss: 11.7785
Epoch [1/10], Batch [15/603], Loss: 10.3314
Epoch [1/10], Batch [16/603], Loss: 15.7068
Epoch [1/10], Batch [17/603], Loss: 13.4503
Epoch [1/10], Batch [18/603], Loss: 10.7750
Epoch [1/10], Batch [19/603], Loss: 14.6776
Epoch [1/10], Batch [20/603], Loss: 16.5842
Epoch [1/10], Batch [21/603], Loss: 9.0457
Epoch [1/10], Batch [22/603], Loss: 14.2302
Epoch [1/10], Batch [23/603], Loss: 15.9001

In [35]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [36]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for data, target in val_loader:  # валидационный даталоадер
        data = {k: v.to(device) for k, v in data.items()}
        target = target.to(device)
        output = model(data)
        preds = torch.argmax(output, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

acc = accuracy_score(all_targets, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro')
conf_mat = confusion_matrix(all_targets, all_preds)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Confusion Matrix:\n{conf_mat}")

Accuracy: 0.9851
Precision: 0.9668
Recall: 0.9853
F1-score: 0.9747
Confusion Matrix:
[[ 16   0   0   0   0   0   0   0   0   0   0   0   0]
 [  0  19   0   0   0   0   0   0   0   0   0   0   0]
 [  0   0  19   0   0   0   0   0   0   0   0   0   0]
 [  0   0   0 258   7   0   0   0   0   0   0   0   0]
 [  0   0   0   0  60   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0  67   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   4   0   0   0   0   0   0]
 [  0   0   0   0   0   2   1 116   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0  54   2   0   0   0]
 [  0   0   0   0   0   0   0   0   5 375   0   0   0]
 [  0   0   0   0   0   0   0   1   0   0  10   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0 162   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0  27]]


In [37]:
model.eval()  

val_loss_total = 0.0
num_batches = 0

with torch.no_grad():  
    for data, target in val_loader:
        data = {k: v.to(device) for k, v in data.items()}
        target = target.to(device)

        output = model(data)
        loss = criterion(output, target)

        val_loss_total += loss.item()
        num_batches += 1

val_loss_avg = val_loss_total / num_batches
print(f"Validation Loss: {val_loss_avg:.4f}")

Validation Loss: 0.0719


In [38]:
torch.save(model, "bert_for_classifing_orders.pth")